# VIN Decoder API - Usage Examples

This notebook demonstrates how to use the NHTSA vPIC VIN Decoder API from Jupyter.

## Prerequisites

Make sure the API is running:
```bash
cd services/vin-api
docker-compose up -d
./setup-database.sh  # or setup-database.bat on Windows
```

In [1]:
# Import required libraries
import requests
import pandas as pd
import json
from typing import Dict, List

In [2]:
# API Configuration
API_URL = "http://localhost:8000"

# Test if API is accessible
try:
    response = requests.get(f"{API_URL}/health")
    print(f"✅ API Status: {response.json()}")
except Exception as e:
    print(f"❌ Error connecting to API: {e}")
    print("Make sure the API is running with: docker-compose up -d")

✅ API Status: {'status': 'healthy', 'database': 'connected', 'message': 'API is running and database is accessible'}


## Example 1: Decode a Single VIN

In [3]:
# Decode a sample VIN
vin = "1HGBH41JXMN109186"

response = requests.get(f"{API_URL}/decode/{vin}")

if response.status_code == 200:
    data = response.json()
    print(f"VIN: {data['vin']}")
    print(f"Total records: {data['count']}")
    print(f"Success: {data['success']}")
    print(f"\nFirst 10 attributes:")
    for item in data['data'][:10]:
        print(f"  {item['variable']}: {item['value']}")
else:
    print(f"Error: {response.status_code} - {response.text}")

VIN: 1HGBH41JXMN109186
Total records: 19
Success: True

First 10 attributes:
  Vehicle Descriptor: 1HGBH41J*MN
  Suggested VIN: 
  Error Code: 8
  Error Text: 8 - No detailed data available currently
  Additional Error Text: None
  Possible Values: 
  Vehicle Type: PASSENGER CAR
  Manufacturer Name: AMERICAN HONDA MOTOR CO., INC.
  Model Year: 1991
  Make: HONDA


## Example 2: Extract Specific Vehicle Information

In [4]:
def get_vehicle_info(vin: str) -> Dict[str, str]:
    """Decode VIN and extract key vehicle information."""
    response = requests.get(f"{API_URL}/decode/{vin}")
    
    if response.status_code != 200:
        return {"error": f"Failed to decode VIN: {response.status_code}"}
    
    data = response.json()
    
    # Convert list of dicts to a single dict
    vehicle_data = {item['variable']: item['value'] for item in data['data']}
    
    # Extract key fields
    return {
        'VIN': vin,
        'Make': vehicle_data.get('Make', 'N/A'),
        'Model': vehicle_data.get('Model', 'N/A'),
        'Year': vehicle_data.get('Model Year', 'N/A'),
        'Body Class': vehicle_data.get('Body Class', 'N/A'),
        'Engine': vehicle_data.get('Engine Model', 'N/A'),
        'Manufacturer': vehicle_data.get('Manufacturer Name', 'N/A'),
        'Plant': vehicle_data.get('Plant City', 'N/A'),
    }

# Test the function
vehicle_info = get_vehicle_info("1HGBH41JXMN109186")
print(json.dumps(vehicle_info, indent=2))

{
  "VIN": "1HGBH41JXMN109186",
  "Make": "HONDA",
  "Model": "N/A",
  "Year": "1991",
  "Body Class": "N/A",
  "Engine": "N/A",
  "Manufacturer": "AMERICAN HONDA MOTOR CO., INC.",
  "Plant": "N/A"
}


## Example 3: Decode Multiple VINs

In [ ]:
# Sample VINs to decode
sample_vins = [
    "1HGBH41JXMN109186",  # Honda Accord
    "2HGFG12668H542570",  # Honda Civic
    "1FTFW1EF8DFC10312",  # Ford F-150
    "5YJSA1E14HF212444",  # Tesla Model S
    "1G1YY25U865112345",  # Chevrolet Corvette
]

# Decode all VINs
results = []
for vin in sample_vins:
    info = get_vehicle_info(vin)
    results.append(info)
    print(f"✓ Decoded: {vin}")

# Convert to DataFrame
df = pd.DataFrame(results)
print("\nDecoded Vehicles:")
df

## Example 4: Get Full Vehicle Details

In [6]:
# Get complete details for a VIN
vin = "1HGBH41JXMN109186"
response = requests.get(f"{API_URL}/decode/{vin}")

if response.status_code == 200:
    data = response.json()
    
    # Convert to DataFrame for better viewing
    df_details = pd.DataFrame(data['data'])
    
    # Filter out empty values
    df_details = df_details[df_details['value'].notna()]
    df_details = df_details[df_details['value'] != '']
    
    print(f"Complete VIN Decode for: {vin}")
    print(f"Total attributes: {len(df_details)}")
    print("\nAll Attributes:")
    df_details.head(20)  # Show first 20 attributes

Complete VIN Decode for: 1HGBH41JXMN109186
Total attributes: 16

All Attributes:


## Example 5: Error Handling

In [ ]:
# Test with invalid VIN (too short)
invalid_vin = "ABC123"

response = requests.get(f"{API_URL}/decode/{invalid_vin}")
print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Example 6: Using POST Method

In [ ]:
# Decode VIN using POST request
vin = "1HGBH41JXMN109186"

response = requests.post(
    f"{API_URL}/decode",
    json={"vin": vin}
)

if response.status_code == 200:
    data = response.json()
    vehicle_info = {item['variable']: item['value'] for item in data['data']}
    
    print(f"VIN: {data['vin']}")
    print(f"Make: {vehicle_info.get('Make')}")
    print(f"Model: {vehicle_info.get('Model')}")
    print(f"Year: {vehicle_info.get('Model Year')}")

## Example 7: Batch Processing with Progress

In [ ]:
from tqdm import tqdm  # pip install tqdm if not available

# Large list of VINs
vins_to_decode = [
    "1HGBH41JXMN109186",
    "2HGFG12668H542570",
    "1FTFW1EF8DFC10312",
    "5YJSA1E14HF212444",
    # Add more VINs here...
]

results = []
for vin in tqdm(vins_to_decode, desc="Decoding VINs"):
    try:
        info = get_vehicle_info(vin)
        results.append(info)
    except Exception as e:
        print(f"Error decoding {vin}: {e}")
        results.append({"VIN": vin, "error": str(e)})

# Create DataFrame
df_batch = pd.DataFrame(results)
print(f"\nSuccessfully decoded {len(df_batch)} VINs")
df_batch.head()

## Example 8: Export Results

In [ ]:
# Export to CSV
if 'df' in locals():
    output_file = "decoded_vehicles.csv"
    df.to_csv(output_file, index=False)
    print(f"✅ Results exported to {output_file}")

# Export to Excel
# df.to_excel("decoded_vehicles.xlsx", index=False)

# Export to JSON
# df.to_json("decoded_vehicles.json", orient="records", indent=2)

## Useful Helper Functions

In [ ]:
def validate_vin(vin: str) -> bool:
    """Validate VIN format (basic check)."""
    return len(vin) == 17 and vin.isalnum()

def decode_vin_safe(vin: str) -> Dict:
    """Decode VIN with error handling."""
    if not validate_vin(vin):
        return {"error": "Invalid VIN format. Must be 17 alphanumeric characters."}
    
    try:
        return get_vehicle_info(vin)
    except Exception as e:
        return {"error": str(e)}

# Test
print("Valid VIN:", decode_vin_safe("1HGBH41JXMN109186"))
print("\nInvalid VIN:", decode_vin_safe("ABC123"))